# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**The contract, in five plain answers** *(my lane: Lane 2, Refresh / Content Opportunity Scoring; month used: `month=2026-03`)*

1. **What one row means.** In the warehouse table, one row = one content item, for one client, on one day (`report_date` x `client_hash_id` x `content_hash_id`). In my feature frame, one row = one content item (client + content) summarised over the first half of March.
2. **Table I use.** `fact_content_daily_performance`, reading only the `month=2026-03` partition. I do not use the `_sample` table for label logic, and I do not touch the final month (June 2026), which stays sealed as a test month.
3. **Time window.** Features come only from 2026-03-01 to 2026-03-15. The label is computed from 2026-03-16 to 2026-03-30 (15 days each side, so the two halves are the same length).
4. **What I predict.** `is_declining` = 1 when a page's impressions in the second half are below 80% of its first-half impressions. This is a proxy for "worth an editor's review", not an observed outcome. The model ranks pages for the review queue.
5. **One thing I deliberately exclude.** Everything computed from the second half of March (`imp_h2` and anything derived from it). Those are the label's own ingredients, so they can never be features.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `imp_h1`, `clk_h1`, `ctr_h1`, `days_with_imp_h1`, `avg_pos_h1` | Built only from rows dated 2026-03-01 to 2026-03-15 (first half) |
| **Label / proxy** | `is_declining` (from `imp_h2` vs `imp_h1`) | The thing I rank on. `imp_h2` is the label's ingredient |
| **Context** | `client_hash_id`, `content_hash_id` | Joining and grouped train/test splits only. Never a feature |
| **Excluded** | `imp_h2` and any column derived from it | Future information. It is the label |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Setup.** The Hugging Face token comes from the Colab Secret named `HF_TOKEN` (or a hidden prompt). It is never typed into a cell, because this repo is public.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn
import os, getpass
import duckdb, pandas as pd

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face READ token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL  = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

**Query 1 of 3 - grain.** Claim: one row really is one `report_date` x `client_hash_id` x `content_hash_id`. If that holds, the number of repeated keys is 0.

In [ ]:
q1 = con.sql(f"""
    SELECT COUNT(*) AS repeated_grain_keys
    FROM (
        SELECT report_date, client_hash_id, content_hash_id
        FROM {FACT}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").df()
q1

**Query 2 of 3 - row count and date span.** Claim: this slice is the March 2026 partition, from 2026-03-01 to 2026-03-31.

In [ ]:
q2 = con.sql(f"""
    SELECT COUNT(*)                        AS n_rows,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {FACT}
""").df()
q2

**Query 3 of 3 - availability.** Rows before a client's GA4 start are zero-filled with `ga4_data_available = FALSE`, so I filter with `IS TRUE` and count the survivors.

In [ ]:
q3 = con.sql(f"""
    SELECT COUNT(*)                                            AS n_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS n_rows_ga4_available,
           ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_surviving
    FROM {FACT}
""").df()
q3

**What the three outputs say** *(fill in from your own run, in plain words)*

- Grain: `[repeated_grain_keys = ?]`, so the grain `[holds / does not hold]`.
- Row count and span: `[n_rows]` rows from `[first_day]` to `[last_day]`, across `[n_clients]` clients and `[n_content_items]` content items.
- Availability: `[n_rows_ga4_available]` of `[n_rows]` rows survive `IS TRUE` (`[pct_surviving]%`).
- Note: my five features use only GSC columns, so the GA4 flag does not change the feature frame. A GA4 feature would need this filter first.

**Five features, max.** One row per content item. Pages with under 100 first-half impressions are dropped, because their rates are too noisy to read.

In [ ]:
frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_h1,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END)      AS clk_h1,
           COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0
                               THEN report_date END)                                        AS days_with_imp_h1,
           AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END)       AS avg_pos_h1,
           SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_h2
    FROM {FACT}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-30'
    GROUP BY 1, 2
    HAVING imp_h1 >= 100
""").df()

frame['ctr_h1'] = frame['clk_h1'] / frame['imp_h1']
frame['is_declining'] = (frame['imp_h2'] < 0.8 * frame['imp_h1']).astype(int)

FEATURES = ['imp_h1', 'clk_h1', 'ctr_h1', 'days_with_imp_h1', 'avg_pos_h1']
print(f"{len(frame):,} content items | declining rate: {frame['is_declining'].mean():.3f}")
frame[FEATURES + ['is_declining']].describe().T

**Knowable at the decision moment because...** *(the decision moment is the end of 2026-03-15)*

| Feature | Knowable at the decision moment because... |
|---|---|
| `imp_h1` | it sums impressions from rows dated 2026-03-01 to 2026-03-15 only |
| `clk_h1` | it sums clicks from the same first-half rows only |
| `ctr_h1` | it is `clk_h1 / imp_h1`, so it uses nothing after 2026-03-15 |
| `days_with_imp_h1` | it counts first-half days with impressions, all dated on or before 2026-03-15 |
| `avg_pos_h1` | it averages the daily position over first-half rows only |

**The trap.** First the honest score: a quick model, split by client so no client appears on both sides.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def quick_score(cols):
    data = frame.dropna(subset=cols)
    X, y, groups = data[cols], data['is_declining'], data['client_hash_id']
    train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(X, y, groups))
    model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    model.fit(X.iloc[train], y.iloc[train])
    auc = roc_auc_score(y.iloc[test], model.predict_proba(X.iloc[test])[:, 1])
    base = max(y.iloc[test].mean(), 1 - y.iloc[test].mean())
    return round(auc, 3), round(base, 3)

honest_auc, base_rate = quick_score(FEATURES)
print(f"HONEST   ROC AUC = {honest_auc}   (always-guess-majority accuracy = {base_rate})")

Now add ONE label-derived column on purpose: the second-half impressions as a ratio of the first half. This is the label's own ingredient, so the model is being shown the answer.

In [ ]:
frame['LEAK_h2_over_h1'] = frame['imp_h2'] / frame['imp_h1']

leaky_auc, _ = quick_score(FEATURES + ['LEAK_h2_over_h1'])
print(f"WITH LEAK ROC AUC = {leaky_auc}")
if leaky_auc > 0.95:
    print("Suspiciously close to perfect. That is the leak, not skill.")

Delete the leak and keep the honest number.

In [ ]:
frame = frame.drop(columns=['LEAK_h2_over_h1'])
honest_auc, base_rate = quick_score(FEATURES)
print(f"HONEST   ROC AUC = {honest_auc}   (this is the number I keep)")
print('columns now:', [c for c in frame.columns if c.startswith('LEAK')] or 'no leak column left')

**What happened** *(fill in from your own run)*

- Honest score: `[honest_auc]`, against a majority-guess accuracy of `[base_rate]`.
- With the leak: `[leaky_auc]`. A jump like that on one added column is the warning sign: the column held the answer.
- I deleted the column and kept `[honest_auc]` as my result.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**One named limitation of my slice: the label is a proxy built inside one month.** It compares two halves of March 2026, so a "decline" can come from day-of-week mix or ordinary month-shape noise instead of a page actually losing interest. It is not a true next-month outcome, and I have not tested it on the sealed final month. I can say pages "showed a lower second half in this window". I cannot say they "will decline".

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.